© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git

%cd MaskArchitectureAnomaly_CourseProject/eomt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys # Allows importing from the 'eomt' folder
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')

In [4]:
!python3 -m pip install -r requirements.txt

## Setup

In [5]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint

seed_everything(0, verbose=False)

device = 0  # TODO: change to the GPU you want to use
IGNORE_INDEX = 255
img_idx = 10  # TODO: change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/CourseProjectAnomaly"  # drive folder of the cityscapes val set

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "r") as f:
    config_cs = yaml.safe_load(f)

with open("configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml", "r") as f:
    config_coco = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [6]:
# blocco per gestire la doppia configurazione
use_coco = True # Metto True per il modello COCO, False per quello Cityscapes
num_classes_final = 19  # classi di cityscapes

# Scegliamo il config e il file dei pesi in base alla scelta sopra
if use_coco:
    current_config = config_coco
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
    target_img_size = (640, 640)
    #num_classes_to_load = 133
else:
    current_config = config_cs
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"
    #num_classes_to_load = data.num_classes # Usa le classi del dataset (solitamente 19 o 20)
    target_img_size = data.img_size



data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=2,
    num_workers=4,
    check_empty_targets=True,
    img_size = target_img_size,
    **data_module_kwargs
)
data.setup()

## Load model

In [7]:





warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_final,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

# model = (
#    lit_cls(
#         img_size=target_img_size,
#        num_classes=num_classes_final,
#         network=network,
#         **model_kwargs,
#     )
#     .to(device)
# )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load pre-trained weights from Hugging Face Hub
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [8]:
# fix per il parametro mancante sulla size
# Se use_coco è True, forziamo 640. Altrimenti proviamo a leggere dal config
# o usiamo quello di Cityscapes (data.img_size).
if use_coco:
    target_img_size = (640, 640)
else:
    # Prova a prenderlo dal config, se non c'è usa quello del dataset (1024)
    target_img_size = data.img_size

#model_kwargs_final = {k: v for k, v in model_kwargs.items()}
# Ripuliamo i kwargs new
model_kwargs_final = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}
model_kwargs_final.pop("num_classes", None) # Rimuoviamo per non avere sovrascritture
model_kwargs_final.pop("img_size", None)

# FIX CRUCIALE: Rimuoviamo eventuali 'num_classes' dai kwargs che potrebbero sovrascrivere il nostro valore
# if "num_classes" in model_kwargs_final:
#   del model_kwargs_final["num_classes"]

# new
if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs_final["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
is_dinov3 = "dinov3" in name if name else False

if is_dinov3:
    model_kwargs["ckpt_path"] = state_dict_path
    model_kwargs["delta_weights"] = True

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_final,
        network=network,
        **model_kwargs_final,
    )
    .to(device)
)

## Transfer Learning and Fine Tuning

In [9]:
if not is_dinov3:
    state_dict = torch.load(
        state_dict_path, map_location=f"cuda:{device}", weights_only=True
    )

    # Congeliamo tutto e sblocchiamo solo la testa di predizione
    if use_coco:
      keys_to_delete = [
                "network.class_head.weight",
                "network.class_head.bias",
                "criterion.empty_weight"
            ]
      for key in keys_to_delete:
          if key in state_dict:
              del state_dict[key]

model.load_state_dict(state_dict, strict=False)


_IncompatibleKeys(missing_keys=['network.class_head.weight', 'network.class_head.bias', 'criterion.empty_weight'], unexpected_keys=[])

In [10]:
if use_coco:
    for param in model.parameters():
        param.requires_grad = False

    for param in model.network.class_head.parameters():
        param.requires_grad = True

    # checkpoint
    checkpoint_callback = ModelCheckpoint(
      dirpath="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints", # Cartella di salvataggio
      filename="coco-finetuned-cityscapes-{epoch:02d}", # Nome del file
      save_last=True,     # Salva sempre l'ultima epoca
      save_top_k=1,       # Quanti dei "migliori" modelli salvare
      monitor="val/loss", # La metrica da osservare per capire qual è il migliore (puoi usare "val/mIoU" se il tuo modello la calcola)
      mode="min"          # Se usi "val/loss" vuoi il minimo. Se usi "val/mIoU" metti "max"
    )

    # fine tuning dei parametri sbloccati
    trainer = L.Trainer(
        max_epochs=5,              # 10 Numero epoche suggerito
        accelerator="gpu",
        devices=1,
        precision="16-mixed",       # tip del progetto
        callbacks=[checkpoint_callback],
        log_every_n_steps=10,
        default_root_dir="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints"
    )

    # Avvio Fine-tuning
    trainer.fit(model, datamodule=data)

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/CourseProjectAnomaly/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate num

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric PanopticQuality was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)


Training: |          | 0/? [00:00<?, ?it/s]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
IGNORE_INDEX = 255

# new DEFINIZIONE MAPPATURA (COCO -> CITYSCAPES)
# Da compilare con gli ID corretti per il tuo confronto
coco_to_cityscapes_map = {
    # FORMATO -> ID_SEQUENZIALE_COCO: TRAIN_ID_CITYSCAPES

    # --- PEOPLE ---
    0: 11,   # person -> person
    11: 12,  # bicycle (con rider) -> rider (in COCO spesso rider e bike sono legati)

    # --- VEHICLES ---
    1: 18,   # bicycle -> bicycle
    2: 13,   # car -> car
    3: 17,   # motorcycle -> motorcycle
    5: 15,   # bus -> bus
    6: 16,   # train -> train
    7: 14,   # truck -> truck

    # --- OBJECTS ---
    9: 6,    # traffic light -> traffic light
    11: 7,   # stop sign -> traffic sign

    # --- STUFF (Sfondi) ---
    # Questi sono estratti dalla parte finale del CLASS_MAPPING (Stuff classes)
    123: 0,  # road-merged -> road
    124: 1,  # sidewalk-merged -> sidewalk
    125: 2,  # building-merged -> building
    126: 3,  # wall-merged -> wall
    127: 4,  # fence-merged -> fence
    128: 5,  # pole-merged -> pole
    129: 8,  # vegetation-merged -> vegetation
    130: 9,  # terrain-merged -> terrain
    131: 10, # sky-merged -> sky
}

# new
def translate_preds(pred_array):
    """Converte i label COCO in label Cityscapes"""
    translated = np.full_like(pred_array, fill_value=IGNORE_INDEX)
    for coco_id, city_id in coco_to_cityscapes_map.items():
        translated[pred_array == coco_id] = city_id
    return translated

def infer_semantic(img, target):
    model.eval()

    # FIX  PER LE FINESTRE
    # Forza il modello a usare la dimensione della finestra corretta
    # (640 per COCO, 1024 per Cityscapes)
    model.window_size = target_img_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()

    # Se il modello caricato è COCO (134 classi), traduciamo i risultati
    if use_coco: # se siamo nel modello coco
        pred_array = translate_preds(pred_array)

    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[
        0
    ].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Gestione dell'immagine (se normalizzata o meno)
    img_np = img.permute(1, 2, 0).cpu().numpy()
    # Se i valori sono fuori dal range [0,1], normalizziamo per la visualizzazione
    if img_np.max() > 1.0 or img_np.min() < 0:
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# run for the cityscapes pre-trained model
img_cs, target_cs = data.val_dataloader().dataset[img_idx]
pred_array_cs, target_array_cs = infer_semantic(img_cs, target_cs)
plot_semantic_results(img_cs, pred_array_cs, target_array_cs)

In [ ]:
# run for the coco pre-trained model
img_cc, target_cc = data.val_dataloader().dataset[img_idx]
pred_array_cc, target_array_cc = infer_semantic(img_cc, target_cc)
plot_semantic_results(img_cc, pred_array_cc, target_array_cc)

## Evaluation
Consistent evaluation strategy that allows fair comparison across both models for the semantic segmentation task

In [ ]:
import numpy as np
from tqdm import tqdm

def fast_hist(a, b, n):
    """
    Calcola la matrice di confusione.
    a: ground truth (target), b: predizione, n: numero di classi
    """
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n) # Modifica qui: filtra anche b
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    """Calcola la IoU per ogni classe dalla matrice di confusione."""
    return np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist))

# setting
num_eval_classes = 19 # Cityscapes ha 19 classi di valutazione
hist = np.zeros((num_eval_classes, num_eval_classes))
val_loader = data.val_dataloader()

print(f"evaluation on {len(val_loader)} images...")
print(f"Current model: {'COCO' if use_coco else 'Cityscapes'}")

# evaluation
for i, batch in enumerate(tqdm(val_loader)):
    # Il dataloader restituisce (immagine, target)
    # Il batch è nella forma [[immagine_tensor], [target_dict]]
    # Dobbiamo estrarre gli elementi effettivi dalle liste
    img = batch[0][0]
    target = batch[1][0]

    # Eseguiamo l'inferenza semantica (che include già la mappatura per COCO)
    pred_array, target_array = infer_semantic(img, target)

    # Appiattiamo gli array per il calcolo della matrice
    # Nota: target_array contiene già i train_id (0-18) e 255 per ignore
    hist += fast_hist(target_array.flatten(), pred_array.flatten(), num_eval_classes)

# calcolo finale
ious = per_class_iu(hist)
miou = np.nanmean(ious)

# visualization results
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("\n" + "="*30)
print(f"Results MIOU - {'COCO' if use_coco else 'CITYSCAPES'}")
print("="*30)
for name, iou in zip(classes, ious):
    print(f"{name:15}: {iou*100:6.2f}%")
print("-" * 30)
print(f"MEAN IoU: {miou*100:6.2f}%")
print("="*30)